# Part 1

Đây là file notebook được sử dụng để kiểm chứng các hàm đã được cài đặt trong `part1`. 


Các hàm sẽ được kiểm chứng bao gồm:
- `gaussian_eliminate(A, b)`
- `back_substitution(U, c)`
- `determinant(A)`
- `inverse(A)`
- `rank_and_basis(A)`


## Import các hàm 

In [2]:
import numpy as np
from gaussian import gaussian_eliminate
from back_substitution import back_substitution
from determinant import determinant
from inverse import inverse
from rank_basis import rank_and_basis
from utils import verify_solution

## Kiểm nghiệm `gaussian_eliminate` 

In [3]:
A_tests = [
    [[1, 2, 3], [2, 7, 6], [1, 2, 8]],
    [[0, 2], [3, -1]],
    [[1, 1], [1, 1]],
    [[1, 2, 3], [2, 4, 6], [1, 1, 1]],
    [[1e-16, 1], [1, 1]],
]

b_tests = [
    [[5], [8], [4]],
    [[4], [5]],
    [[2], [3]],
    [[4], [8], [3]],
    [[1], [2]],
]

passed = 0

for i in range(len(A_tests)):
    U, x, num_swaps = gaussian_eliminate(A_tests[i].copy(), b_tests[i].copy())

    if x is None:
        rank_A = np.linalg.matrix_rank(A_tests[i])
        rank_aug = np.linalg.matrix_rank(np.c_[A_tests[i], b_tests[i]])

        if rank_A < rank_aug:
            print("Numpy check: PASSED (no_solution)")
            passed += 1
        else:
            print("Numpy check: FAILED (should not be None)")
        continue

    is_valid, system_type = verify_solution(A_tests[i], x, b_tests[i])

    if is_valid:
        print(f"Numpy check: PASSED ({system_type})")
        passed += 1
    else:
        print(f"Numpy check: FAILED ({system_type})")
        
print(f"Accuracy: {float(passed / len(A_tests))}")

Numpy check: PASSED (unique_solution)
Numpy check: PASSED (unique_solution)
Numpy check: PASSED (no_solution)
Numpy check: PASSED (infinite_solutions)
Numpy check: PASSED (unique_solution)
Accuracy: 1.0


## Kiểm nghiệm `back_substistution`

In [4]:
U_tests = [
    [[4, -1, 0, 2], [0, 3, 1, 1], [0, 0, 2, 3], [0, 0, 0, 1]],
    [[2, 1], [0, 3]],
    [[1, 2, 3], [0, 4, 5], [0, 0, 6]],
    [[1, 2], [0, 0]],
    [[0, 0], [0, 0]],
]

c_tests = [
    [[6], [7], [4], [2]],
    [[5], [6]],
    [[14], [23], [18]],
    [[3], [1]],
    [[1], [0]],
]

passed = 0

for i in range(len(U_tests)):
    U = [row[:] for row in U_tests[i]]
    c = [row[:] for row in c_tests[i]]

    res = back_substitution(U, c)

    if res is None:
        rank_U = np.linalg.matrix_rank(np.array(U, dtype=float))
        rank_aug = np.linalg.matrix_rank(np.c_[np.array(U, dtype=float), np.array(c, dtype=float)])

        if rank_U < rank_aug:
            print("Numpy check: PASSED (no_solution)")
            passed += 1
        else:
            print("Numpy check: FAILED (should not be None)")
        continue

    is_valid, system_type = verify_solution(U_tests[i], res, c_tests[i])

    if is_valid:
        print(f"Numpy check: PASSED ({system_type})")
        passed += 1
    else:
        print(f"Numpy check: FAILED ({system_type})")

print(f"Accuracy: {float(passed / len(U_tests))}")

Numpy check: PASSED (unique_solution)
Numpy check: PASSED (unique_solution)
Numpy check: PASSED (unique_solution)
Numpy check: PASSED (no_solution)
Numpy check: PASSED (no_solution)
Accuracy: 1.0


## Kiểm nghiệm `determinant`

In [5]:
A_tests = [
    [[1, 2], [3, 4]],
    [[1, 2, 3], [0, 4, 5], [0, 0, 6]],
    [[1, 2], [2, 4]],  # singular
    [[1e-16, 0], [0, 1]],  # small value
    [[1, 2, 3], [4, 5, 6]],  # non-triangular
]

passed = 0

for i in range(len(A_tests)):
    A = [row[:] for row in A_tests[i]]
    res = determinant([row[:] for row in A])

    try:
        np_det = float(np.linalg.det(np.array(A, dtype=float)))
    except Exception:
        np_det = None

    if res is None:
        if np_det is None:
            print("Numpy check: PASSED (None expected)")
            passed += 1
        else:
            print("Numpy check: FAILED (determinant returned None but numpy computed a value)")
        continue

    if np_det is None:
        print("Numpy check: FAILED (numpy could not compute det)")
        continue

    if np.isclose(res, np_det, atol=1e-8, rtol=1e-6):
        print(f"Numpy check: PASSED (det={res})")
        passed += 1
    else:
        print(f"Numpy check: FAILED (mine={res}, numpy={np_det})")

print(f"Accuracy: {float(passed / len(A_tests))}")

Numpy check: PASSED (det=-2.0)
Numpy check: PASSED (det=24.0)
Numpy check: PASSED (det=0.0)
Numpy check: PASSED (det=0.0)
Numpy check: PASSED (None expected)
Accuracy: 1.0


## Kiểm nghiệm `inverse`

In [7]:
A_tests = [
    [[1, 2], [3, 4]],
    [[1, 2, 3], [0, 4, 5], [0, 0, 6]],
    [[1, 2], [2, 4]],  # singular
    [[1e-10, 0], [0, 1]],  # small value
    [[0, 0], [0, 0]],  # zero matrix
    [[1, 2, 3], [4, 5, 6]],  # non-square
]

passed = 0

for i in range(len(A_tests)):
    A = [row[:] for row in A_tests[i]]
    res = inverse([row[:] for row in A])

    try:
        np_inv = np.linalg.inv(np.array(A, dtype=float))
    except Exception:
        np_inv = None

    if res is None:
        if np_inv is None:
            print("Numpy check: PASSED (None expected)")
            passed += 1
        else:
            print("Numpy check: FAILED (inverse returned None but numpy computed a value)")
        continue

    if np_inv is None:
        print("Numpy check: FAILED (numpy could not compute inv)")
        continue

    if np.allclose(np.array(res, dtype=float), np_inv, atol=1e-8, rtol=1e-6):
        print("Numpy check: PASSED (inverse matches)")
        passed += 1
    else:
        print("Numpy check: FAILED (mine vs numpy mismatch)")

print(f"Accuracy: {float(passed / len(A_tests))}")

Numpy check: PASSED (inverse matches)
Numpy check: PASSED (inverse matches)
Numpy check: PASSED (None expected)
Numpy check: PASSED (inverse matches)
Numpy check: PASSED (None expected)
Numpy check: PASSED (None expected)
Accuracy: 1.0


## Kiểm nghiệm `rank_and_basis`

In [3]:
A_tests = [
    [[1, 2], [3, 4]],
    [[1, 2, 3], [0, 4, 5], [0, 0, 6]],
    [[1, 2], [2, 4]],  # rank 1
    [[0, 0], [0, 0]],  # zero matrix
    [[1, 2, 3], [4, 5, 6]],  # 2x3
    [[1, 2, 3], [2, 4, 6], [3, 6, 9]],  # rank 1
]

passed = 0

for i in range(len(A_tests)):
    A = np.array(A_tests[i], dtype=float)

    rank, row_basis, col_basis, null_basis = rank_and_basis([row[:] for row in A_tests[i]])

    np_rank = np.linalg.matrix_rank(A)

    # quick rank check
    if rank != np_rank:
        print(f"Numpy check: FAILED (rank mine={rank}, numpy={np_rank})")
        continue

    # row_basis should have rank equal to matrix rank
    if len(row_basis) > 0 and np.linalg.matrix_rank(np.array(row_basis, dtype=float)) != rank:
        print("Numpy check: FAILED (row_basis rank mismatch)")
        continue

    # col_basis should span the column space of A
    if len(col_basis) > 0:
        C = np.array(col_basis, dtype=float).T  # shape (nr, r)
        ok = True
        for j in range(A.shape[1]):
            if C.size == 0:
                if not np.allclose(A[:, j], 0, atol=1e-8, rtol=1e-6):
                    ok = False
                    break
            else:
                coef, *_ = np.linalg.lstsq(C, A[:, j], rcond=None)
                residual = A[:, j] - C @ coef
                if not np.allclose(residual, 0, atol=1e-8, rtol=1e-6):
                    ok = False
                    break
        if not ok:
            print("Numpy check: FAILED (col_basis does not span columns)")
            continue

    # null_basis vectors should be in nullspace and independent
    NB = np.array(null_basis, dtype=float) if len(null_basis) > 0 else np.zeros((0, A.shape[1]))
    if NB.size > 0:
        ok = True
        for v in NB:
            if not np.allclose(A @ v, np.zeros(A.shape[0]), atol=1e-8, rtol=1e-6):
                ok = False
                break
        if not ok:
            print("Numpy check: FAILED (null_basis vectors are not in nullspace)")
            continue
        # dimension check
        if NB.shape[0] != A.shape[1] - rank:
            print("Numpy check: FAILED (null_basis dimension mismatch)")
            continue
        if NB.shape[0] > 0 and np.linalg.matrix_rank(NB) != NB.shape[0]:
            print("Numpy check: FAILED (null_basis not independent)")
            continue

    print("Numpy check: PASSED")
    passed += 1

print(f"Accuracy: {float(passed / len(A_tests))}")


Numpy check: PASSED
Numpy check: PASSED
Numpy check: PASSED
Numpy check: PASSED
Numpy check: PASSED
Numpy check: PASSED
Accuracy: 1.0
